In [7]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'stylizer').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import soundfile as sf
from IPython.display import Audio
from stylizer.cfm_lightning_module import CFMLightningModule


In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CFMLightningModule.load_for_inference(str(REPO_ROOT / 'assets/ckpts/stylizer-no-style-enc.ckpt'), device=device)


Using HuggingFace SSL encoder from transformers


In [ ]:
def convert(model, src_path, tgt_path, steps=16, cfg_strength=2, save=False):
    """
    src_path can be either a .wav path or a folder containing a source .wav.
    tgt_path must be a target-style folder containing both a .wav and a .npy style embedding.
    """
    out_dict = model.sample(
        cond=tgt_path,
        content=src_path,
        steps=steps,
        cfg_strength=cfg_strength
    )
    pred_audio = out_dict['pred_audio'].squeeze().cpu().numpy()
    if save:
        sf.write('converted.wav', pred_audio, 16000)
    
    return Audio(pred_audio, rate=16000)


In [ ]:
convert(
    model,
    src_path=str(REPO_ROOT / 'assets/target_examples/happy'),
    tgt_path=str(REPO_ROOT / 'assets/target_examples/british'),
    steps=16,
    cfg_strength=2,
    save=True,
)
